# UniMap Corpus Inspection

In [1]:
import pickle, random
from collections import Counter

PKL_PATH = '/home/tommy/Projects/PCBSDA/experiment/outputs/raw_data/cross-architecture/unimap/normalized_corpus.pkl'
with open(PKL_PATH, 'rb') as f:
    corpus = pickle.load(f)

for arch, blocks in corpus.items():
    vocab = set(tok for b in blocks for tok in b.split())
    print(f'{arch}: {len(blocks):,} blocks, {len(vocab):,} unique tokens')

x86_64: 6,215,043 blocks, 80,066 unique tokens
arm_32: 5,233,121 blocks, 61,533 unique tokens
mips_32: 6,564,690 blocks, 105,227 unique tokens


## 1. Format check — raw samples

In [2]:
# Show 10 random normalized blocks per arch
N = 10
for arch, blocks in corpus.items():
    print(f'\n=== {arch} ===')
    for b in random.sample(blocks, N):
        toks = b.split()
        print(f'  [{len(toks):2d}]  ' + '  '.join(toks))


=== x86_64 ===
  [ 3]  MOV~RAX,[RBP-0]  MOV~RAX,[RAX+0]  JMP~0
  [ 4]  MOV~ESI,0  MOV~EDI,0  CALL~0  JMP~0
  [ 7]  MOV~RAX,[RBP-0]  MOV~EDI,0  MOV~RSI,[RAX+0]  CALL~0  MOV~EDX,0  MOV~ESI,0  MOV~R12,RAX
  [ 2]  TEST~CL,CL  JZ~0
  [ 2]  MOV~EAX,0  JMP~0
  [ 2]  CMP~[RBP-0],0  JZ~0
  [ 1]  JMP~0
  [ 1]  JMP~0
  [ 4]  ADD~R13,0  MOV~RAX,[RBP+0]  CMP~[RAX+R14*0+0],R13D  JG~0
  [ 6]  MOV~RDX,RBX  MOV~RSI,R12  MOV~RDI,RAX  CALL~0  CMP~R12,RAX  JNZ~0

=== arm_32 ===
  [ 3]  LDR~R6,[0]  LDR~R4,[0]  B~0
  [10]  LDR~R3,[R11,0]  LDR~R3,[R3,0]  LDR~R3,[R3,0]  SUB~R2,R11,0  LDR~R1,[R11,0]  LDR~R0,[R11,0]  BLX~R3  LDR~R3,[R11,0]  CMN~R3,0  BEQ~0
  [ 1]  LDR~R10,[R13,0]
  [ 2]  TEQ~R4,R12  BNE~0
  [12]  LDRB~R3,[R11,0]  CMP~R3,0  LDRNE~R3,[R13,0]  STRNE~R3,[R4,0]  LDRNE~R3,[R4,0]  ADDNE~R3,R3,0  STRNE~R3,[R4,0]  MOVEQ~R3,0  STREQ~R3,[R4,0]  MOV~R3,0  STR~R3,[R4,0]  B~0
  [ 5]  LDR~R3,[R13,0]  LDR~R2,[R13,0]  SUBS~R3,R3,R2  MOVNE~R3,0  B~0
  [ 3]  MOV~R3,0  MOV~R6,0  STRB~R3,[R8,0]
  [ 2]  CMP~R3,0  B

## 2. Vocab inflation diagnosis — tokens that appear only once (hapax)

In [3]:
# Hapax legomena (count==1) are the main driver of vocab inflation.
# If normalization is correct, there should be very few.
for arch, blocks in corpus.items():
    counter = Counter(tok for b in blocks for tok in b.split())
    hapax = [t for t, c in counter.items() if c == 1]
    rare  = [t for t, c in counter.items() if c <= 5]
    print(f'\n=== {arch} ===')
    print(f'  Total unique: {len(counter):,}')
    print(f'  Hapax (count=1): {len(hapax):,}  ({100*len(hapax)/len(counter):.1f}% of vocab)')
    print(f'  Rare  (count≤5): {len(rare):,}  ({100*len(rare)/len(counter):.1f}% of vocab)')
    print('  Sample hapax:')
    for t in hapax[:20]:
        print(f'    {t}')


=== x86_64 ===
  Total unique: 80,066
  Hapax (count=1): 19,429  (24.3% of vocab)
  Rare  (count≤5): 43,200  (54.0% of vocab)
  Sample hapax:
    MOV~R13,[RBP+R14*0]
    LEA~R14,[R13+R12*0-0]
    LEA~R12,[R9+RDX*0-0]
    MOVSX~RAX,[RAX+R13*0]
    MOV~[R11+R9*0+0],R8D
    MOVZX~R8D,[RSI+RSI*0+0]
    LEA~R11,[R9+R11*0+0]
    CMP~[RCX+R14*0+0],0
    MOVSXD~RSI,[RDI-0]
    MOV~[R9+RAX*0],R8
    MOVD~XMM2,[RAX+0]
    MOVD~XMM4,[RAX+0]
    MOVD~XMM0,[RAX-0]
    MOVD~XMM5,[RAX-0]
    CMOVBE~R12,[R14+0]
    MOVD~[RDX+RAX*0],XMM0
    MOV~[RDX+RAX*0],BP
    MOV~[RDX+RAX*0],R12W
    CMP~[RBX+RAX*0-0],R14
    MOV~[R14+R15*0],RBX

=== arm_32 ===
  Total unique: 61,533
  Hapax (count=1): 14,301  (23.2% of vocab)
  Rare  (count≤5): 32,909  (53.5% of vocab)
  Sample hapax:
    LDRBHI~R3,[R7,R3]
    STRBHI~R3,[R14],0
    SUBLT~R6,R1,0
    CMPGE~R3,R6
    CPYGE~R12,R9
    RSBPL~R14,R5,0
    ORR~R14,R5,R7
    ANDS~R3,R5,R1
    CPYGE~R12,R7
    ORR~R14,R4,R6
    SBCS~R3,R10,R7
    MULLE~R9,R10,R9
    BIC

## 3. Token structure breakdown — what part causes unique inflation

In [4]:
import re

def classify_token(tok):
    if '~' not in tok:
        return 'opcode_only'
    opcode, rest = tok.split('~', 1)
    # Check if any operand part still contains a raw number
    if re.search(r'(?<![A-Z<])0x[0-9a-fA-F]+', rest):
        return 'has_raw_hex'
    if re.search(r'(?<![A-Z<>\[,*+\-])\b[0-9]{2,}\b', rest):
        return 'has_raw_decimal'
    return 'normalized'

for arch, blocks in corpus.items():
    counter = Counter(tok for b in blocks for tok in b.split())
    cats = Counter(classify_token(t) for t in counter)
    print(f'\n=== {arch} ===')
    for cat, cnt in cats.most_common():
        print(f'  {cat}: {cnt:,} unique tokens')

    # Show examples of leaking tokens
    leaked = [(t, counter[t]) for t in counter if classify_token(t) in ('has_raw_hex', 'has_raw_decimal')]
    leaked.sort(key=lambda x: -x[1])
    if leaked:
        print(f'  Top leaking tokens (raw numbers not normalized):')
        for t, c in leaked[:10]:
            print(f'    {t!r:60s}  count={c}')


=== x86_64 ===
  normalized: 80,030 unique tokens
  opcode_only: 36 unique tokens

=== arm_32 ===
  normalized: 61,532 unique tokens
  opcode_only: 1 unique tokens

=== mips_32 ===
  normalized: 105,192 unique tokens
  opcode_only: 35 unique tokens


## 4. Embedded PKL Inspection — (150, 200) MAIE vectors

In [2]:
import pickle
import numpy as np
import pandas as pd
from collections import Counter

PKL_BASE = '/home/tommy/Projects/PCBSDA/experiment/outputs/embedded_graphs/cross-architecture/Unimap/x86_64_arm_32'
CSV_PATH = '/home/tommy/Projects/PCBSDA/datasets/csv/cross_architecture_dataset.csv'

archs = ['x86_64', 'arm_32']
embeds = {}

for arch in archs:
    with open(f'{PKL_BASE}/{arch}.pkl', 'rb') as f:
        embeds[arch] = pickle.load(f)

df = pd.read_csv(CSV_PATH)
sha_to_family = dict(zip(df['file_name'], df['family']))

print('='*60)
for arch in archs:
    d = embeds[arch]
    shapes = [v.shape for v in d.values()]
    assert all(s == (150, 200) for s in shapes), "Shape mismatch!"

    # OOV check: entries where every row is zero
    all_zero = sum(1 for v in d.values() if np.all(v == 0))
    # Partially OOV: rows that are zero within each matrix
    zero_rows = sum((v == 0).all(axis=1).sum() for v in d.values())
    total_rows = len(d) * 150

    # Value range
    sample = next(iter(d.values()))
    all_vals = np.concatenate([v.flatten() for v in list(d.values())[:50]])

    print(f'\n[{arch}]')
    print(f'  Entries       : {len(d):,}')
    print(f'  Shape per entry: {shapes[0]}  ✓')
    print(f'  All-zero entries (fully OOV): {all_zero}')
    print(f'  Zero token rows : {zero_rows:,} / {total_rows:,}  ({100*zero_rows/total_rows:.1f}% OOV tokens)')
    print(f'  Value range (first 50 entries): [{all_vals.min():.4f}, {all_vals.max():.4f}]')
    print(f'  Sample[0] first 5 dims: {sample[0, :5]}')

    # Family distribution
    families = [sha_to_family.get(sha, 'UNKNOWN') for sha in d.keys()]
    fam_cnt = Counter(families)
    print(f'  Family distribution:')
    for fam, cnt in sorted(fam_cnt.items(), key=lambda x: -x[1]):
        print(f'    {fam:20s}: {cnt:4d}')

print('\n' + '='*60)
print('Format OK — all entries are (150, 200) float32')



[x86_64]
  Entries       : 2,073
  Shape per entry: (150, 200)  ✓
  All-zero entries (fully OOV): 0
  Zero token rows : 6,764 / 310,950  (2.2% OOV tokens)
  Value range (first 50 entries): [-0.3112, 0.2902]
  Sample[0] first 5 dims: [-0.057287   -0.0545086   0.00243718  0.034904    0.0190257 ]
  Family distribution:
    mobidash            :  300
    gafgyt              :  300
    meterpreter         :  300
    tsunami             :  300
    mirai               :  300
    kaiji               :  240
    dnsamp              :  208
    dofloo              :  125

[arm_32]
  Entries       : 1,814
  Shape per entry: (150, 200)  ✓
  All-zero entries (fully OOV): 0
  Zero token rows : 38,859 / 272,100  (14.3% OOV tokens)
  Value range (first 50 entries): [-0.3072, 0.3281]
  Sample[0] first 5 dims: [-0.0649454  -0.12113     0.0427212  -0.00083258 -0.0853409 ]
  Family distribution:
    dofloo              :  300
    mobidash            :  300
    mirai               :  300
    tsunami        

In [2]:
import json
file_path = "/home/tommy/Projects/nict_reverse_script_bk/reverse/Unimap/output/results/arm_32/a2ps-4.14_gcc-10.3.0_arm_32_O0_a2ps.json"

function = json.load(open(file_path))
for blocks in function.values():
    print(f'Function Name: {list(function.keys())[list(function.values()).index(blocks)]}')
    print(f'Function with {len(blocks)} blocks:')
    for block in blocks:
        print(block)

#check basic block amount

count = 0
for blocks in function.values():
    count += len(blocks)
print(f'Total basic blocks: {count}')

Function Name: _init
Function with 1 blocks:
stmdb sp!,{r3,lr} bl 0x00011e5c ldmia sp!,{r3,pc}
Function Name: _start
Function with 1 blocks:
mov r11,#0x0 mov lr,#0x0 ldr r1,[sp],#0x4 cpy r2,sp str r2,[sp,#-0x4]! str r0,[sp,#-0x4]! ldr r12,[0x11e50] str r12,[sp,#-0x4]! ldr r0,[0x11e54] ldr r3,[0x11e58] bl 0x00011bbc bl 0x00011dd8
Function Name: call_weak_fn
Function with 3 blocks:
adr r12,0x11bf4 add r12,r12,#0x79000 ldr pc,[r12,#0x4e4]!
ldr r3,[0x11e78] ldr r2,[0x11e7c] add r3,pc,r3 ldr r2,[r3,r2] cmp r2,#0x0 bxeq lr
b 0x00011bec
Function Name: deregister_tm_clones
Function with 3 blocks:
ldr r0,[0x11ea0] ldr r3,[0x11ea4] cmp r3,r0 bxeq lr
ldr r3,[0x11ea8] cmp r3,#0x0 bxeq lr
bx r3
Function Name: register_tm_clones
Function with 3 blocks:
ldr r0,[0x11ed8] ldr r1,[0x11edc] sub r3,r1,r0 mov r1,r3, lsr #0x1f add r1,r1,r3, asr #0x2 movs r1,r1, asr #0x1 bxeq lr
ldr r3,[0x11ee0] cmp r3,#0x0 bxeq lr
bx r3
Function Name: __do_global_dtors_aux
Function with 2 blocks:
stmdb sp!,{r4,lr} ldr r4,[0

In [ ]:
# check unique tokens in the raw corpus

file_path = "/home/tommy/Projects/nict_reverse_script_bk/reverse/Unimap/corpus_norm/x86_64_corpus_norm.txt"

with open(file_path, 'r') as f:
    lines = f.readlines()
vocab = set(tok for line in lines for tok in line.split())
print("x86_64")
print(f'Unique tokens in raw corpus: {len(vocab):,}')

X86_64
Unique tokens in raw corpus: 82,038


In [9]:
# check unique tokens in the raw corpus

file_path = "/home/tommy/Projects/nict_reverse_script_bk/reverse/Unimap/corpus_norm/arm_32_corpus_norm.txt"

with open(file_path, 'r') as f:
    lines = f.readlines()
vocab = set(tok for line in lines for tok in line.split())
print("ARM-32")
print(f'Unique tokens in raw corpus: {len(vocab):,}')

ARM-32
Unique tokens in raw corpus: 79,167


In [5]:
file_path = "/home/tommy/Projects/nict_reverse_script_bk/reverse/Unimap/corpus_norm/mips_32_corpus_norm.txt"

with open(file_path, 'r') as f:
    lines = f.readlines()
vocab = set(tok for line in lines for tok in line.split())
print("MIPS-32")
print(f'Unique tokens in raw corpus: {len(vocab):,}')

MIPS-32
Unique tokens in raw corpus: 82,462


In [7]:
import fasttext
import numpy as np

model = fasttext.load_model("/home/tommy/Projects/PCBSDA/experiment/cross-architecture/unimap/embeddings/x86_64.bin")

def sim(w1, w2):
    v1, v2 = model.get_word_vector(w1), model.get_word_vector(w2)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return float('nan')
    return float(np.dot(v1, v2) / (n1 * n2))

def top_neighbors(token, k=5):
    words = model.get_words()
    q = model.get_word_vector(token)
    scores = [(w, sim(token, w)) for w in words]
    scores.sort(key=lambda x: -x[1])
    return scores[1:k+1]  # skip self

# ── 1. 語意相近 vs 無關 ──────────────────────────────────────────────────────
print("=== 語意相近 vs 無關 ===")
pairs = [
    # 預期高相似
    ("MOV~RAX,QWORD~PTR~[RBP+-0]", "MOV~ESI,0",     "MOV 系列"),
    ("ADD~RSP,0",                   "SUB~RSP,0",      "stack 調整 (ADD/SUB)"),
    ("JZ~<FOO>",                    "JNZ~<FOO>",      "條件跳轉"),
    ("PUSH~RBP",                    "POP~RBP",        "PUSH/POP 同 reg"),
    # 預期低相似
    ("MOV~EAX,0",                   "JMP~<FOO>",      "資料移動 vs 跳轉"),
    ("CALL~<FOO>",                  "XOR~EAX,EAX",    "呼叫 vs 清零"),
]
for w1, w2, label in pairs:
    print(f"  {label:25s}  {sim(w1, w2):+.4f}  ({w1[:30]}  ↔  {w2[:30]})")

# ── 2. Nearest neighbors ────────────────────────────────────────────────────
print("\n=== Nearest neighbors ===")
queries = ["MOV~EAX,0", "CALL~<FOO>", "JZ~<FOO>", "XOR~EAX,EAX"]
for q in queries:
    neighbors = top_neighbors(q, k=5)
    print(f"\n  [{q}]")
    for w, s in neighbors:
        print(f"    {s:+.4f}  {w}")


=== 語意相近 vs 無關 ===
  MOV 系列                     +0.3296  (MOV~RAX,QWORD~PTR~[RBP+-0]  ↔  MOV~ESI,0)
  stack 調整 (ADD/SUB)         +0.3944  (ADD~RSP,0  ↔  SUB~RSP,0)
  條件跳轉                       +0.8871  (JZ~<FOO>  ↔  JNZ~<FOO>)
  PUSH/POP 同 reg             +0.2897  (PUSH~RBP  ↔  POP~RBP)
  資料移動 vs 跳轉                 +0.4155  (MOV~EAX,0  ↔  JMP~<FOO>)
  呼叫 vs 清零                   +0.4642  (CALL~<FOO>  ↔  XOR~EAX,EAX)

=== Nearest neighbors ===

  [MOV~EAX,0]
    +0.5618  MOV~EDX,0
    +0.5440  XOR~EAX,EAX
    +0.5258  MOV~ESI,0
    +0.5174  MOVSX~ESI,R15B
    +0.5113  MOV~WORD~PTR~[RBX],DI

  [CALL~<FOO>]
    +0.7471  MOV~ESI,0
    +0.7072  MOV~RDI,RAX
    +0.6599  MOVSX~R8D,R15B
    +0.6452  LEA~R13,[RAX+RBP*0+0]
    +0.6405  MOV~RSI,QWORD~PTR~[R12+R14*0+-0]

  [JZ~<FOO>]
    +0.8871  JNZ~<FOO>
    +0.5962  </s>
    +0.5238  AND~AL,BL
    +0.5199  MOVSXD~R12,DWORD~PTR~[R15]
    +0.5089  LEA~RAX,[RCX+RDI*0+-0]

  [XOR~EAX,EAX]
    +0.6074  MOV~ESI,0
    +0.5922  MOVSX~R8D,R15B
    +0.572

## MIPS-32 Embedding Quality Test

In [8]:
import fasttext
import numpy as np

model_mips = fasttext.load_model("/home/tommy/Projects/PCBSDA/experiment/cross-architecture/unimap/embeddings/mips_32.bin")

def sim_mips(w1, w2):
    v1, v2 = model_mips.get_word_vector(w1), model_mips.get_word_vector(w2)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return float('nan')
    return float(np.dot(v1, v2) / (n1 * n2))

def top_neighbors_mips(token, k=5):
    words = model_mips.get_words()
    scores = [(w, sim_mips(token, w)) for w in words]
    scores.sort(key=lambda x: -x[1])
    return scores[1:k+1]

# ── 1. 語意相近 vs 無關 ──────────────────────────────────────────────────────
print("=== 語意相近 vs 無關 ===")
pairs = [
    # 預期高相似
    ("LW~R2,<OFF>R30",    "LW~R25,<OFF>R28",   "LW 同類 load"),
    ("SW~R2,<OFF>R29",    "SW~R31,<OFF>R29",   "SW 同類 store"),
    ("ADD~R29,R29,0",     "ADD~R29,R29,-0",    "stack 調整 (ADD imm)"),
    ("BEQ~R2,R0,<FOO>",   "BNE~R2,R0,<FOO>",  "條件跳轉 BEQ/BNE"),
    ("JAL~<FOO>",         "JALR~R25",          "函式呼叫 JAL/JALR"),
    # 預期低相似
    ("LW~R2,<OFF>R30",    "BEQ~R2,R0,<FOO>",  "load vs 條件跳轉"),
    ("JAL~<FOO>",         "LI~R2,0",           "呼叫 vs 立即數載入"),
]
for w1, w2, label in pairs:
    print(f"  {label:28s}  {sim_mips(w1, w2):+.4f}  ({w1}  ↔  {w2})")

# ── 2. Nearest neighbors ────────────────────────────────────────────────────
print("\n=== Nearest neighbors ===")
queries = ["LW~R2,<OFF>R30", "JAL~<FOO>", "BEQ~R2,R0,<FOO>", "OR~R2,R0,R0"]
for q in queries:
    neighbors = top_neighbors_mips(q, k=5)
    print(f"\n  [{q}]")
    for w, s in neighbors:
        print(f"    {s:+.4f}  {w}")


=== 語意相近 vs 無關 ===
  LW 同類 load                    +0.2349  (LW~R2,<OFF>R30  ↔  LW~R25,<OFF>R28)
  SW 同類 store                   +0.4056  (SW~R2,<OFF>R29  ↔  SW~R31,<OFF>R29)
  stack 調整 (ADD imm)            +0.3412  (ADD~R29,R29,0  ↔  ADD~R29,R29,-0)
  條件跳轉 BEQ/BNE                  +0.9350  (BEQ~R2,R0,<FOO>  ↔  BNE~R2,R0,<FOO>)
  函式呼叫 JAL/JALR                 +0.6024  (JAL~<FOO>  ↔  JALR~R25)
  load vs 條件跳轉                  +0.2415  (LW~R2,<OFF>R30  ↔  BEQ~R2,R0,<FOO>)
  呼叫 vs 立即數載入                   +0.4110  (JAL~<FOO>  ↔  LI~R2,0)

=== Nearest neighbors ===

  [LW~R2,<OFF>R30]
    +0.8405  LW~R3,<OFF>R30
    +0.8404  SW~R2,<OFF>R30
    +0.7550  LW~R28,<OFF>R30
    +0.7531  LW~R4,<OFF>R30
    +0.7410  OR~R25,R2,R0

  [JAL~<FOO>]
    +0.6730  BAL~<FOO>
    +0.6303  SEB~R5,R17
    +0.6218  OR~R6,R2,R18
    +0.6174  OR~R4,R16,R0
    +0.6148  ADD~R16,R18,R21

  [BEQ~R2,R0,<FOO>]
    +0.9350  BNE~R2,R0,<FOO>
    +0.5514  MOVZ~R5,R19,R2
    +0.5437  MOVZ~R5,R21,R2
    +0.5270  BGTZ~R2,<FOO>

## ARM-32 Embedding Quality Test

In [ ]:
import fasttext
import numpy as np

model_arm = fasttext.load_model("/home/tommy/Projects/PCBSDA/experiment/cross-architecture/unimap/embeddings/arm_32.bin")

def sim_arm(w1, w2):
    v1, v2 = model_arm.get_word_vector(w1), model_arm.get_word_vector(w2)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return float('nan')
    return float(np.dot(v1, v2) / (n1 * n2))

def top_neighbors_arm(token, k=5):
    words = model_arm.get_words()
    scores = [(w, sim_arm(token, w)) for w in words]
    scores.sort(key=lambda x: -x[1])
    return scores[1:k+1]

# ── 1. 語意相近 vs 無關 ──────────────────────────────────────────────────────
print("=== 語意相近 vs 無關 ===")
pairs = [
    # 預期高相似
    ("LDR~R3,[R11,-0]",   "LDR~R3,[R3,0]",    "LDR 同類 load"),
    ("STR~R3,[R13,0]",    "STR~R3,[R11,-0]",  "STR 同類 store"),
    ("ADD~R13,R13,0",     "SUB~R13,R13,0",    "stack 調整 (ADD/SUB SP)"),
    ("BEQ~<FOO>",         "BNE~<FOO>",        "條件跳轉 BEQ/BNE"),
    ("BL~<FOO>",          "BX~R14",           "函式呼叫/返回 BL/BX"),
    # 預期低相似
    ("LDR~R3,[R11,-0]",   "BEQ~<FOO>",        "load vs 條件跳轉"),
    ("BL~<FOO>",          "MOV~R3,0",         "呼叫 vs 立即數載入"),
]
for w1, w2, label in pairs:
    print(f"  {label:28s}  {sim_arm(w1, w2):+.4f}  ({w1}  ↔  {w2})")

# ── 2. Nearest neighbors ────────────────────────────────────────────────────
print("\n=== Nearest neighbors ===")
queries = ["LDR~R3,[R11,-0]", "BL~<FOO>", "BEQ~<FOO>", "MOV~R3,0"]
for q in queries:
    neighbors = top_neighbors_arm(q, k=5)
    print(f"\n  [{q}]")
    for w, s in neighbors:
        print(f"    {s:+.4f}  {w}")


## x86_64 → MIPS-32 Mapping Quality Test

In [16]:
import numpy as np
from collections import defaultdict

EMB_DIR = "/home/tommy/Projects/PCBSDA/experiment/cross-architecture/unimap/embeddings/x86_64_mips_32"

def load_vec(path):
    words, vecs = [], []
    with open(path, encoding='utf-8') as f:
        n, dim = map(int, f.readline().split())
        for line in f:
            parts = line.rstrip().split(' ')
            words.append(parts[0])
            vecs.append(list(map(float, parts[1:])))
    return words, np.array(vecs, dtype=np.float32)

print("Loading mapped embeddings ...")
x86_words, X = load_vec(f"{EMB_DIR}/x86_64_mapped.vec")
mips_words, M = load_vec(f"{EMB_DIR}/mips_32_mapped.vec")

x86_idx  = {w: i for i, w in enumerate(x86_words)}
mips_idx = {w: i for i, w in enumerate(mips_words)}

# L2 normalize for cosine sim via dot product
X_n = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
M_n = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)

def cross_sim(x86_tok, mips_tok):
    i, j = x86_idx.get(x86_tok), mips_idx.get(mips_tok)
    if i is None or j is None:
        return float('nan')
    return float(X_n[i] @ M_n[j])

def cross_neighbors(x86_tok, k=5):
    i = x86_idx.get(x86_tok)
    if i is None:
        return []
    scores = X_n[i] @ M_n.T          # (n_mips,)
    top = np.argsort(scores)[::-1][:k]
    return [(mips_words[j], float(scores[j])) for j in top]

# ── 1. seed pairs — mapping 後語意等價的 pair 相似度應該高 ──────────────────
print("=== Seed pair cross-arch similarity (應該高) ===")
seed_pairs = [
    ("CALL~<FOO>",                      "JAL~<FOO>"),
    ("RET",                             "JR~R31"),
    ("JMP~<FOO>",                       "B~<FOO>"),
    ("JZ~<FOO>",                        "BEQ~R2,R0,<FOO>"),
    ("JNZ~<FOO>",                       "BNE~R2,R0,<FOO>"),
    ("SUB~RSP,0",                       "ADD~R29,R29,-0"),
    ("ADD~RSP,0",                       "ADD~R29,R29,0"),
    ("XOR~EAX,EAX",                     "OR~R2,R0,R0"),
    ("MOV~EAX,0",                       "LI~R2,0"),
    ("MOV~RAX,QWORD~PTR~[RBP+-0]",      "LW~R2,<OFF>R30"),
    ("MOV~QWORD~PTR~[RBP+-0],RAX",      "SW~R2,<OFF>R30"),
    ("PUSH~RBP",                        "SW~R30,<OFF>R29"),
    ("POP~RBP",                         "LW~R30,<OFF>R29"),
    ("MOV~RBP,RSP",                     "OR~R30,R29,R0"),
]
for x, m in seed_pairs:
    s = cross_sim(x, m)
    print(f"  {s:+.4f}  {x[:38]:38s}  ↔  {m}")

# ── 2. non-pair — 語意無關的 pair 相似度應該低 ────────────────────────────────
print("\n=== Non-seed cross-arch similarity (應該低) ===")
non_pairs = [
    ("CALL~<FOO>",    "LW~R2,<OFF>R30"),
    ("JMP~<FOO>",     "SW~R2,<OFF>R29"),
    ("XOR~EAX,EAX",  "JAL~<FOO>"),
    ("PUSH~RBP",      "BEQ~R2,R0,<FOO>"),
]
for x, m in non_pairs:
    s = cross_sim(x, m)
    print(f"  {s:+.4f}  {x[:38]:38s}  ↔  {m}")

# ── 3. x86 token 的跨架構 nearest neighbors ───────────────────────────────
print("\n=== x86_64 → MIPS cross-arch nearest neighbors ===")
queries = ["CALL~<FOO>", "JZ~<FOO>", "MOV~EAX,0", "PUSH~RBP", "XOR~EAX,EAX"]
for q in queries:
    neighbors = cross_neighbors(q, k=5)
    print(f"\n  x86: [{q}]")
    for w, s in neighbors:
        print(f"    {s:+.4f}  mips: {w}")


Loading mapped embeddings ...
=== Seed pair cross-arch similarity (應該高) ===
  +0.5991  CALL~<FOO>                              ↔  JAL~<FOO>
  +0.5503  RET                                     ↔  JR~R31
  +0.6060  JMP~<FOO>                               ↔  B~<FOO>
  +0.2175  JZ~<FOO>                                ↔  BEQ~R2,R0,<FOO>
  +0.1564  JNZ~<FOO>                               ↔  BNE~R2,R0,<FOO>
  +0.6052  SUB~RSP,0                               ↔  ADD~R29,R29,-0
  +0.4651  ADD~RSP,0                               ↔  ADD~R29,R29,0
  +0.2163  XOR~EAX,EAX                             ↔  OR~R2,R0,R0
  +0.2626  MOV~EAX,0                               ↔  LI~R2,0
  +0.7597  MOV~RAX,QWORD~PTR~[RBP+-0]              ↔  LW~R2,<OFF>R30
  +0.7115  MOV~QWORD~PTR~[RBP+-0],RAX              ↔  SW~R2,<OFF>R30
  +0.6113  PUSH~RBP                                ↔  SW~R30,<OFF>R29
  +0.4508  POP~RBP                                 ↔  LW~R30,<OFF>R29
  +0.6057  MOV~RBP,RSP                             ↔  

## x86_64 → ARM-32 Mapping Quality Test

In [19]:
import numpy as np

EMB_DIR = "/home/tommy/Projects/PCBSDA/experiment/cross-architecture/unimap/embeddings/x86_64_arm_32"

def load_vec(path):
    words, vecs = [], []
    with open(path, encoding='utf-8') as f:
        n, dim = map(int, f.readline().split())
        for line in f:
            parts = line.rstrip().split(' ')
            words.append(parts[0])
            vecs.append(list(map(float, parts[1:])))
    return words, np.array(vecs, dtype=np.float32)

print("Loading mapped embeddings ...")
x86_words, X = load_vec(f"{EMB_DIR}/x86_64_mapped.vec")
arm_words,  A = load_vec(f"{EMB_DIR}/arm_32_mapped.vec")

x86_idx = {w: i for i, w in enumerate(x86_words)}
arm_idx  = {w: i for i, w in enumerate(arm_words)}

X_n = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
A_n = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)

def cross_sim(x86_tok, arm_tok):
    i, j = x86_idx.get(x86_tok), arm_idx.get(arm_tok)
    if i is None or j is None:
        return float('nan')
    return float(X_n[i] @ A_n[j])

def cross_neighbors(x86_tok, k=5):
    i = x86_idx.get(x86_tok)
    if i is None:
        return []
    scores = X_n[i] @ A_n.T
    top = np.argsort(scores)[::-1][:k]
    return [(arm_words[j], float(scores[j])) for j in top]

# ── 1. seed pairs 相似度（應該高）────────────────────────────────────────────
print("=== Seed pair cross-arch similarity (應該高) ===")
seed_pairs = [
    ("CALL~<FOO>",                   "BL~<FOO>"),
    ("CALL~QWORD~PTR~[RAX+0]",       "BLX~R3"),
    ("RET",                          "BX~R14"),
    ("JMP~<FOO>",                    "B~<FOO>"),
    ("JZ~<FOO>",                     "BEQ~<FOO>"),
    ("JNZ~<FOO>",                    "BNE~<FOO>"),
    ("JA~<FOO>",                     "BHI~<FOO>"),
    ("JLE~<FOO>",                    "BLE~<FOO>"),
    ("SUB~RSP,0",                    "SUB~R13,R13,0"),
    ("ADD~RSP,0",                    "ADD~R13,R13,0"),
    ("MOV~EAX,0",                    "MOV~R0,0"),
    ("XOR~EAX,EAX",                  "MOV~R0,0"),
    ("MOV~RAX,QWORD~PTR~[RBP+-0]",   "LDR~R3,[R11,-0]"),
    ("MOV~QWORD~PTR~[RBP+-0],RAX",   "STR~R3,[R11,-0]"),
    ("PUSH~RBP",                     "STMDB~R13!~R11"),
    ("POP~RBP",                      "LDMIA~R13!~R11"),
    ("MOV~RBP,RSP",                  "ADD~R11,R13,0"),
    ("CMP~EAX,0",                    "CMP~R3,0"),
]
for x, a in seed_pairs:
    s = cross_sim(x, a)
    print(f"  {s:+.4f}  {x[:38]:38s}  ↔  {a}")

# ── 2. non-pair 相似度（應該低）──────────────────────────────────────────────
print("\n=== Non-seed cross-arch similarity (應該低) ===")
non_pairs = [
    ("CALL~<FOO>",   "LDR~R3,[R11,-0]"),
    ("JMP~<FOO>",    "STR~R3,[R13,0]"),
    ("XOR~EAX,EAX", "BL~<FOO>"),
    ("PUSH~RBP",     "BEQ~<FOO>"),
]
for x, a in non_pairs:
    s = cross_sim(x, a)
    print(f"  {s:+.4f}  {x[:38]:38s}  ↔  {a}")

# ── 3. cross-arch nearest neighbors ─────────────────────────────────────────
print("\n=== x86_64 → ARM-32 cross-arch nearest neighbors ===")
queries = ["CALL~<FOO>", "JZ~<FOO>", "MOV~EAX,0", "PUSH~RBP", "XOR~EAX,EAX"]
for q in queries:
    neighbors = cross_neighbors(q, k=5)
    print(f"\n  x86: [{q}]")
    for w, s in neighbors:
        print(f"    {s:+.4f}  arm: {w}")


Loading mapped embeddings ...
=== Seed pair cross-arch similarity (應該高) ===
  +0.6824  CALL~<FOO>                              ↔  BL~<FOO>
  +0.2725  CALL~QWORD~PTR~[RAX+0]                  ↔  BLX~R3
  +0.3257  RET                                     ↔  BX~R14
  +0.5134  JMP~<FOO>                               ↔  B~<FOO>
  +0.6970  JZ~<FOO>                                ↔  BEQ~<FOO>
  +0.6253  JNZ~<FOO>                               ↔  BNE~<FOO>
  +0.5324  JA~<FOO>                                ↔  BHI~<FOO>
  +0.4441  JLE~<FOO>                               ↔  BLE~<FOO>
  +0.6140  SUB~RSP,0                               ↔  SUB~R13,R13,0
  +0.5141  ADD~RSP,0                               ↔  ADD~R13,R13,0
  +0.3882  MOV~EAX,0                               ↔  MOV~R0,0
  +0.4568  XOR~EAX,EAX                             ↔  MOV~R0,0
  +0.7609  MOV~RAX,QWORD~PTR~[RBP+-0]              ↔  LDR~R3,[R11,-0]
  +0.6799  MOV~QWORD~PTR~[RBP+-0],RAX              ↔  STR~R3,[R11,-0]
  +0.6782  PUSH~RBP

## MIPS-32 → ARM-32 Mapping Quality Test

In [9]:
import numpy as np

EMB_DIR = "/home/tommy/Projects/PCBSDA/experiment/cross-architecture/unimap/embeddings/mips_32_arm_32"

def load_vec(path):
    words, vecs = [], []
    with open(path, encoding='utf-8') as f:
        n, dim = map(int, f.readline().split())
        for line in f:
            parts = line.rstrip().split(' ')
            words.append(parts[0])
            vecs.append(list(map(float, parts[1:])))
    return words, np.array(vecs, dtype=np.float32)

print("Loading mapped embeddings ...")
mips_words, M = load_vec(f"{EMB_DIR}/mips_32_mapped.vec")
arm_words,  A = load_vec(f"{EMB_DIR}/arm_32_mapped.vec")

mips_idx = {w: i for i, w in enumerate(mips_words)}
arm_idx  = {w: i for i, w in enumerate(arm_words)}

M_n = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)
A_n = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)

def cross_sim(mips_tok, arm_tok):
    i, j = mips_idx.get(mips_tok), arm_idx.get(arm_tok)
    if i is None or j is None:
        return float('nan')
    return float(M_n[i] @ A_n[j])

def cross_neighbors(mips_tok, k=5):
    i = mips_idx.get(mips_tok)
    if i is None:
        return []
    scores = M_n[i] @ A_n.T
    top = np.argsort(scores)[::-1][:k]
    return [(arm_words[j], float(scores[j])) for j in top]

# ── 1. seed pairs（應該高）───────────────────────────────────────────────────
print("=== Seed pair cross-arch similarity (應該高) ===")
seed_pairs = [
    ("JAL~<FOO>",           "BL~<FOO>"),
    ("BAL~<FOO>",           "BL~<FOO>"),
    ("JALR~R25",            "BL~<FOO>"),
    ("JR~R31",              "LDMIA~R13!~R15"),
    ("B~<FOO>",             "B~<FOO>"),
    ("BEQ~R2,R0,<FOO>",     "BEQ~<FOO>"),
    ("BEQ~R3,R0,<FOO>",     "BEQ~<FOO>"),
    ("BEQ~R4,R0,<FOO>",     "BEQ~<FOO>"),
    ("BNE~R2,R0,<FOO>",     "BNE~<FOO>"),
    ("BNE~R3,R0,<FOO>",     "BNE~<FOO>"),
    ("OR~R2,R0,R0",         "MOV~R0,0"),
    ("OR~R3,R0,R0",         "MOV~R1,0"),
    ("OR~R4,R0,R0",         "MOV~R2,0"),
    ("OR~R2,R3,R0",         "MOV~R0,0"),
    ("OR~R3,R2,R0",         "MOV~R1,0"),
    ("LI~R2,0",             "MOV~R0,0"),
    ("LI~R3,0",             "MOV~R1,0"),
    ("LI~R4,0",             "MOV~R2,0"),
    ("SW~R31,<OFF>R29",     "STMDB~R13!~R14"),
    ("LW~R31,<OFF>R29",     "LDMIA~R13!~R15"),
    ("SW~R30,<OFF>R29",     "STMDB~R13!~R11"),
    ("LW~R30,<OFF>R29",     "LDMIA~R13!~R11"),
    ("LW~R2,<OFF>R30",      "LDR~R0,[R11,-0]"),
    ("SW~R2,<OFF>R30",      "STR~R0,[R11,-0]"),
    ("LW~R2,<OFF>R29",      "LDR~R3,[R13,0]"),
    ("SW~R2,<OFF>R29",      "STR~R3,[R13,0]"),
    ("OR~R30,R29,R0",       "ADD~R11,R13,0"),
    ("SLL~R2,R2,0",         "MOV~R3,R3,LSL~0"),
    ("ADD~R13,R13,0",       "ADD~R13,R13,0"),
    ("SUB~R13,R13,0",       "SUB~R13,R13,0"),
]
for m, a in seed_pairs:
    s = cross_sim(m, a)
    print(f"  {s:+.4f}  {m:<30s}  ↔  {a}")

# ── 2. non-pair（語意無關，應該低）────────────────────────────────────────────
print("\n=== Non-seed cross-arch similarity (應該低) ===")
non_pairs = [
    ("JAL~<FOO>",        "LDR~R0,[R11,-0]"),
    ("B~<FOO>",          "STR~R0,[R11,-0]"),
    ("OR~R2,R0,R0",      "BL~<FOO>"),
    ("SW~R31,<OFF>R29",  "BEQ~<FOO>"),
    ("LW~R2,<OFF>R30",   "B~<FOO>"),
]
for m, a in non_pairs:
    s = cross_sim(m, a)
    print(f"  {s:+.4f}  {m:<30s}  ↔  {a}")

# ── 3. MIPS → ARM cross-arch nearest neighbors ──────────────────────────────
print("\n=== MIPS-32 → ARM-32 cross-arch nearest neighbors ===")
queries = ["JAL~<FOO>", "BEQ~R2,R0,<FOO>", "LW~R2,<OFF>R30", "OR~R2,R0,R0", "SW~R31,<OFF>R29"]
for q in queries:
    neighbors = cross_neighbors(q, k=5)
    print(f"\n  mips: [{q}]")
    for w, s in neighbors:
        print(f"    {s:+.4f}  arm: {w}")


Loading mapped embeddings ...
=== Seed pair cross-arch similarity (應該高) ===
  +0.6248  JAL~<FOO>                       ↔  BL~<FOO>
  +0.6793  BAL~<FOO>                       ↔  BL~<FOO>
  +0.7103  JALR~R25                        ↔  BL~<FOO>
  +0.5786  JR~R31                          ↔  LDMIA~R13!~R15
  +0.5633  B~<FOO>                         ↔  B~<FOO>
  +0.3366  BEQ~R2,R0,<FOO>                 ↔  BEQ~<FOO>
  +0.0880  BEQ~R3,R0,<FOO>                 ↔  BEQ~<FOO>
  +0.1776  BEQ~R4,R0,<FOO>                 ↔  BEQ~<FOO>
  +0.3297  BNE~R2,R0,<FOO>                 ↔  BNE~<FOO>
  +0.1048  BNE~R3,R0,<FOO>                 ↔  BNE~<FOO>
  +0.2303  OR~R2,R0,R0                     ↔  MOV~R0,0
  +0.2183  OR~R3,R0,R0                     ↔  MOV~R1,0
  +0.4705  OR~R4,R0,R0                     ↔  MOV~R2,0
  +0.0851  OR~R2,R3,R0                     ↔  MOV~R0,0
  +0.2460  OR~R3,R2,R0                     ↔  MOV~R1,0
  +0.3224  LI~R2,0                         ↔  MOV~R0,0
  +0.2888  LI~R3,0                

## Mapping Quality Summary — All Pairs

## AUC Evaluation — x86_64 → ARM-32 (similar vs dissimilar pairs)

Paper (UniMap, USENIX'23 §6.3) uses 25 similar + 25 dissimilar x86/ARM pairs from the prior work's test set and reports AUC = 0.78.  
We replicate the same protocol with our MAIE: cosine similarity as score, sklearn roc_auc_score as metric.  
All tokens below are **not in the seed dict** but confirmed present in the mapped .vec vocabulary.

In [16]:
import numpy as np
from sklearn.metrics import roc_auc_score

EMB_DIR = "/home/tommy/Projects/PCBSDA/experiment/cross-architecture/unimap/embeddings/x86_64_arm_32"

def load_vec(path):
    words, vecs = [], []
    with open(path, encoding='utf-8') as f:
        n, dim = map(int, f.readline().split())
        for line in f:
            parts = line.rstrip().split(' ')
            words.append(parts[0])
            vecs.append(list(map(float, parts[1:])))
    return {w: np.array(v, dtype=np.float32) for w, v in zip(words, vecs)}

x86_vecs = load_vec(f"{EMB_DIR}/x86_64_mapped.vec")
arm_vecs  = load_vec(f"{EMB_DIR}/arm_32_mapped.vec")

def cosine(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0: return float('nan')
    return float(np.dot(v1, v2) / (n1 * n2))

# Similar pairs (label=1) — NOT in seeds/x86_64_arm_32.txt
similar = [
    # Control Transfer — indirect call/jump
    ("CALL~RAX",                             "BLX~R3"),
    ("CALL~RDX",                             "BLX~R4"),
    ("CALL~RBX",                             "BLX~R5"),
    ("JMP~RAX",                              "BX~R3"),
    # Conditional branch — conditions not in seed
    ("JGE~<FOO>",                            "BGE~<FOO>"),
    ("JL~<FOO>",                             "BLT~<FOO>"),
    ("JNS~<FOO>",                            "BPL~<FOO>"),
    ("JNC~<FOO>",                            "BCC~<FOO>"),
    ("JC~<FOO>",                             "BCS~<FOO>"),
    ("JO~<FOO>",                             "BVS~<FOO>"),
    # Conditional branch — seeded conditions, different registers
    ("JZ~<FOO>",                             "BEQ~<FOO>"),
    ("JNZ~<FOO>",                            "BNE~<FOO>"),
    ("JA~<FOO>",                             "BHI~<FOO>"),
    ("JBE~<FOO>",                            "BLS~<FOO>"),
    ("JG~<FOO>",                             "BGT~<FOO>"),
    ("JLE~<FOO>",                            "BLE~<FOO>"),
    ("JS~<FOO>",                             "BMI~<FOO>"),
    # Load from stack frame (RBP/R11)
    ("MOV~RDX,QWORD~PTR~[RBP+-0]",          "LDR~R2,[R11,-0]"),
    ("MOV~RCX,QWORD~PTR~[RBP+-0]",          "LDR~R1,[R11,-0]"),
    ("MOV~RSI,QWORD~PTR~[RBP+-0]",          "LDR~R3,[R11,-0]"),
    ("MOV~RDI,QWORD~PTR~[RBP+-0]",          "LDR~R0,[R11,-0]"),
    ("MOV~EAX,DWORD~PTR~[RAX+0]",           "LDR~R3,[R3,0]"),
    ("MOV~EDX,DWORD~PTR~[RBP+-0]",          "LDR~R2,[R11,-0]"),
    # Load from RSP/R13
    ("MOV~RCX,QWORD~PTR~[RSP+0]",           "LDR~R1,[R13,0]"),
    ("MOV~RDX,QWORD~PTR~[RSP+0]",           "LDR~R2,[R13,0]"),
    # Move immediate zero
    ("MOV~EDI,0",                            "MOV~R0,0"),
    ("MOV~ECX,0",                            "MOV~R3,0"),
    ("MOV~R8D,0",                            "MOV~R0,0"),
    ("MOV~R9D,0",                            "MOV~R1,0"),
    # Register-to-register copy
    ("MOV~RDI,RBP",                          "CPY~R0,R4"),
    ("MOV~RDI,RBX",                          "CPY~R0,R5"),
    ("MOV~RSI,RBP",                          "CPY~R1,R4"),
    ("MOV~RDX,RBX",                          "CPY~R2,R3"),
    ("MOV~RAX,RCX",                          "CPY~R0,R1"),
    # Store to stack frame
    ("MOV~DWORD~PTR~[RBP+-0],EAX",          "STR~R3,[R11,-0]"),
    ("MOV~DWORD~PTR~[RBP+-0],EDX",          "STR~R2,[R11,-0]"),
    ("MOV~QWORD~PTR~[RBP+-0],RDI",          "STR~R0,[R11,-0]"),
    ("MOV~QWORD~PTR~[RBP+-0],RSI",          "STR~R3,[R11,-0]"),
    ("MOV~DWORD~PTR~[RSP+0],0",             "STR~R3,[R13,0]"),
    ("MOV~QWORD~PTR~[RSP+0],RCX",           "STR~R1,[R13,0]"),
    # Push/pop callee-saved regs
    ("PUSH~R12",                             "STMDB~R13!~R5"),
    ("POP~R12",                              "LDMIA~R13!~R5"),
    ("PUSH~R13",                             "STMDB~R13!~R6"),
    ("POP~R13",                              "LDMIA~R13!~R6"),
    ("PUSH~R14",                             "STMDB~R13!~R7"),
    ("POP~R14",                              "LDMIA~R13!~R7"),
    ("PUSH~R15",                             "STMDB~R13!~R8"),
    ("POP~R15",                              "LDMIA~R13!~R8"),
    ("PUSH~RAX",                             "STMDB~R13!~R14"),
    ("POP~RDX",                              "LDMIA~R13!~R4"),
    # Arithmetic
    ("SUB~EAX,0",                            "SUB~R3,R3,0"),
    ("SUB~RDX,0",                            "SUB~R2,R2,0"),
    ("SUB~RCX,0",                            "SUB~R1,R1,0"),
    ("ADD~RBX,0",                            "ADD~R3,R3,0"),
    ("ADD~RDX,0",                            "ADD~R2,R2,0"),
    ("ADD~RDX,RAX",                          "ADD~R3,R2,R3"),
    ("ADD~EAX,EDX",                          "ADD~R3,R2,R3"),
    ("ADD~RBP,0",                            "ADD~R11,R13,0"),
    # Compare / Test
    ("TEST~RAX,RAX",                         "CMP~R3,0"),
    ("TEST~EAX,EAX",                         "CMP~R0,0"),
    ("TEST~RDI,RDI",                         "CMP~R0,0"),
    ("TEST~RDX,RDX",                         "CMP~R2,0"),
    ("TEST~RBX,RBX",                         "CMP~R3,0"),
    ("TEST~RSI,RSI",                         "CMP~R3,0"),
    ("CMP~EDX,0",                            "CMP~R2,0"),
    ("CMP~RDX,0",                            "CMP~R2,0"),
    ("CMP~RDX,RAX",                          "CMP~R2,R3"),
    # Logical
    ("AND~EAX,0",                            "AND~R3,R3,0"),
    ("AND~EDX,0",                            "AND~R2,R2,0"),
    ("XOR~EAX,EAX",                          "MOV~R0,0"),
    ("XOR~EDI,EDI",                          "MOV~R0,0"),
    ("XOR~EDX,EDX",                          "MOV~R2,0"),
    ("XOR~ESI,ESI",                          "MOV~R3,0"),
    ("OR~EAX,EDX",                           "ORR~R3,R3,R2"),
    ("OR~EAX,EAX",                           "ORR~R3,R3,0"),
    # Shift
    ("SHL~RAX,0",                            "MOV~R3,R3,LSL~0"),
    ("SHR~RAX,0",                            "MOV~R3,R3,LSR~0"),
    ("SAR~RAX,0",                            "MOV~R3,R3,ASR~0"),
    ("SHL~RDX,0",                            "MOV~R2,R2,LSL~0"),
    ("SHR~EAX,0",                            "MOV~R3,R3,LSR~0"),
    ("SAR~EAX,0",                            "MOV~R3,R3,ASR~0"),
    # Byte load/store
    ("MOVZX~EAX,BYTE~PTR~[RAX]",            "LDRB~R3,[R3,0]"),
    ("MOVZX~EAX,BYTE~PTR~[RAX+0]",          "LDRB~R3,[R3,0]"),
    ("MOVZX~EAX,BYTE~PTR~[RBP+-0]",         "LDRB~R3,[R11,-0]"),
    ("MOVZX~EAX,BYTE~PTR~[RSP+0]",          "LDRB~R3,[R13,0]"),
    ("MOVZX~EDX,BYTE~PTR~[RAX+0]",          "LDRB~R2,[R3,0]"),
    ("MOV~BYTE~PTR~[RSP+0],0",              "STRB~R3,[R13,0]"),
    ("MOV~BYTE~PTR~[RAX+0],DL",             "STRB~R2,[R3,0]"),
    # Prologue / epilogue
    ("PUSH~RBP",                             "STMDB~R13!~R11"),
    ("POP~RBP",                              "LDMIA~R13!~R11"),
    ("PUSH~RBX",                             "STMDB~R13!~R4"),
    ("POP~RBX",                              "LDMIA~R13!~R4"),
    ("MOV~RBP,RSP",                          "ADD~R11,R13,0"),
    ("SUB~RSP,0",                            "SUB~R13,R13,0"),
    ("ADD~RSP,0",                            "ADD~R13,R13,0"),
    # LEA vs pointer arithmetic
    ("LEA~RAX,[RBP+-0]",                     "SUB~R3,R11,0"),
    ("LEA~RAX,[RSP+0]",                      "ADD~R0,R13,0"),
    ("LEA~RDI,[RBP+-0]",                     "SUB~R0,R11,0"),
    ("LEA~RDX,[RBP+-0]",                     "SUB~R2,R11,0"),
    ("LEA~RSI,[RBP+-0]",                     "SUB~R3,R11,0"),
    # Sign-extend
    ("MOVSXD~RDX,EAX",                       "MOV~R2,R3,LSL~0"),
    ("MOVSXD~RDX,EDX",                       "MOV~R2,R2,LSL~0"),
    # Halfword load
    ("MOVZX~EAX,WORD~PTR~[RAX+0]",          "LDRH~R3,[R3,0]"),
    ("MOVZX~EAX,WORD~PTR~[RBP+-0]",         "LDRH~R3,[R11,-0]"),
    # INC/DEC vs ADD/SUB imm
    ("INC~RAX",                              "ADD~R3,R3,0"),
    ("DEC~RAX",                              "SUB~R3,R3,0"),
    ("INC~EAX",                              "ADD~R3,R3,0"),
    ("DEC~EAX",                              "SUB~R3,R3,0"),
    # NEG vs RSB
    ("NEG~RAX",                              "RSB~R3,R3,0"),
    ("NEG~EAX",                              "RSB~R3,R3,0"),
]

# Dissimilar pairs (label=0) — cross-category
dissimilar = [
    # ctrl vs load
    ("CALL~RAX",                             "LDR~R3,[R11,-0]"),
    ("CALL~RDX",                             "LDR~R2,[R11,-0]"),
    ("JGE~<FOO>",                            "STR~R3,[R13,0]"),
    ("JL~<FOO>",                             "LDR~R2,[R11,-0]"),
    ("JA~<FOO>",                             "LDR~R3,[R3,0]"),
    ("JBE~<FOO>",                            "STR~R3,[R11,-0]"),
    ("JNS~<FOO>",                            "LDR~R3,[R3,0]"),
    ("JC~<FOO>",                             "LDR~R0,[R11,-0]"),
    ("JMP~RAX",                              "STR~R3,[R11,-0]"),
    ("JO~<FOO>",                             "LDR~R1,[R11,-0]"),
    # ctrl vs arith
    ("CALL~RBX",                             "CMP~R3,0"),
    ("JGE~<FOO>",                            "ADD~R3,R3,0"),
    ("JL~<FOO>",                             "SUB~R13,R13,0"),
    ("JNZ~<FOO>",                            "ADD~R2,R2,0"),
    ("JZ~<FOO>",                             "SUB~R3,R3,0"),
    # ctrl vs logic
    ("CALL~RAX",                             "AND~R3,R3,0"),
    ("JMP~RAX",                              "ORR~R3,R3,R2"),
    ("JGE~<FOO>",                            "EOR~R3,R3,0"),
    # load vs ctrl
    ("MOV~RDX,QWORD~PTR~[RBP+-0]",          "BEQ~<FOO>"),
    ("MOV~EAX,DWORD~PTR~[RAX+0]",           "BNE~<FOO>"),
    ("MOV~EDI,0",                            "BL~<FOO>"),
    ("MOV~ECX,0",                            "B~<FOO>"),
    ("MOV~RDI,RBP",                          "BX~R14"),
    ("MOV~RSI,QWORD~PTR~[RBP+-0]",          "BGT~<FOO>"),
    ("MOV~RCX,QWORD~PTR~[RSP+0]",           "BLE~<FOO>"),
    # load vs arith
    ("MOV~DWORD~PTR~[RBP+-0],EAX",          "CMP~R0,0"),
    ("MOVZX~EAX,BYTE~PTR~[RAX+0]",          "ADD~R13,R13,0"),
    ("MOV~RDI,RBX",                          "SUB~R3,R3,0"),
    ("MOV~RAX,RCX",                          "ADD~R2,R2,0"),
    # store vs ctrl
    ("PUSH~R12",                             "BEQ~<FOO>"),
    ("POP~R12",                              "BHI~<FOO>"),
    ("PUSH~R13",                             "BLE~<FOO>"),
    ("PUSH~R14",                             "BNE~<FOO>"),
    ("POP~R15",                              "BLT~<FOO>"),
    # store vs arith
    ("PUSH~RAX",                             "CMP~R3,0"),
    ("POP~RDX",                              "ADD~R3,R2,R3"),
    ("MOV~DWORD~PTR~[RSP+0],0",             "SUB~R13,R13,0"),
    ("MOV~QWORD~PTR~[RSP+0],RCX",           "ADD~R11,R13,0"),
    # arith vs ctrl
    ("TEST~RAX,RAX",                         "BL~<FOO>"),
    ("TEST~EAX,EAX",                         "B~<FOO>"),
    ("SHL~RAX,0",                            "BX~R14"),
    ("ADD~RDX,RAX",                          "BEQ~<FOO>"),
    ("SUB~EAX,0",                            "BNE~<FOO>"),
    # arith vs load
    ("TEST~RAX,RAX",                         "LDR~R3,[R11,-0]"),
    ("SUB~EAX,0",                            "STMDB~R13!~R14"),
    ("ADD~RBX,0",                            "LDMIA~R13!~R11"),
    ("SHL~RAX,0",                            "STR~R3,[R11,-0]"),
    ("CMP~EDX,0",                            "LDR~R2,[R11,-0]"),
    # logic vs ctrl
    ("AND~EAX,0",                            "BEQ~<FOO>"),
    ("XOR~EAX,EAX",                          "BL~<FOO>"),
    ("OR~EAX,EDX",                           "B~<FOO>"),
    # logic vs load
    ("AND~EAX,0",                            "LDR~R3,[R11,-0]"),
    ("XOR~EDI,EDI",                          "STR~R3,[R13,0]"),
    ("OR~EAX,EAX",                           "LDR~R0,[R11,-0]"),
    # shift vs ctrl
    ("SHR~RAX,0",                            "BEQ~<FOO>"),
    ("SAR~EAX,0",                            "BNE~<FOO>"),
    ("SHL~RDX,0",                            "BL~<FOO>"),
    # shift vs load
    ("SHR~EAX,0",                            "LDR~R3,[R11,-0]"),
    ("SAR~RAX,0",                            "STR~R3,[R13,0]"),
    # byte vs ctrl
    ("MOVZX~EAX,BYTE~PTR~[RAX]",            "BEQ~<FOO>"),
    ("MOVZX~EDX,BYTE~PTR~[RAX+0]",          "BNE~<FOO>"),
    ("MOV~BYTE~PTR~[RSP+0],0",              "BL~<FOO>"),
    # byte vs arith
    ("MOVZX~EAX,BYTE~PTR~[RBP+-0]",         "CMP~R3,0"),
    ("MOV~BYTE~PTR~[RAX+0],DL",             "ADD~R13,R13,0"),
    # lea vs ctrl
    ("LEA~RAX,[RBP+-0]",                     "BEQ~<FOO>"),
    ("LEA~RDI,[RBP+-0]",                     "BL~<FOO>"),
    # lea vs load
    ("LEA~RSI,[RBP+-0]",                     "LDR~R3,[R11,-0]"),
    ("LEA~RAX,[RSP+0]",                      "STR~R3,[R13,0]"),
    # prologue vs ctrl
    ("PUSH~RBP",                             "BNE~<FOO>"),
    ("POP~RBP",                              "BGT~<FOO>"),
    # prologue vs load
    ("PUSH~RBX",                             "LDR~R3,[R3,0]"),
    ("POP~RBX",                              "STR~R3,[R11,-0]"),
    # misc cross-category
    ("MOVSXD~RDX,EAX",                       "BEQ~<FOO>"),
    ("MOVZX~EAX,WORD~PTR~[RAX+0]",          "ADD~R3,R2,R3"),
    ("LEA~RDX,[RBP+-0]",                     "ORR~R3,R3,R2"),
    ("CMP~RDX,RAX",                          "LDR~R3,[R11,-0]"),
    ("ADD~EAX,EDX",                          "BLE~<FOO>"),
    ("AND~EDX,0",                            "STMDB~R13!~R6"),
    ("XOR~ECX,ECX",                          "LDR~R1,[R13,0]"),
    ("SAR~RAX,0",                            "STMDB~R13!~R5"),
    ("SHR~EAX,0",                            "BHI~<FOO>"),
    ("MOV~R8D,0",                            "B~<FOO>"),
    ("MOVZX~EAX,WORD~PTR~[RBP+-0]",         "CMP~R3,0"),
    ("OR~EAX,EDX",                           "LDR~R2,[R11,-0]"),
    ("INC~RAX",                              "BEQ~<FOO>"),
    ("NEG~RAX",                              "BL~<FOO>"),
    ("DEC~EAX",                              "STR~R3,[R13,0]"),
    ("MOVSXD~RDX,EDX",                       "BNE~<FOO>"),
    ("MOVZX~EAX,BYTE~PTR~[RSP+0]",          "ADD~R13,R13,0"),
    ("LEA~RSI,[RBP+-0]",                     "CMP~R2,0"),
    ("NEG~EAX",                              "LDR~R3,[R3,0]"),
    ("INC~EAX",                              "BHI~<FOO>"),
    ("DEC~RAX",                              "STR~R3,[R11,-0]"),
    ("MOVZX~EAX,WORD~PTR~[RBP+-0]",         "BNE~<FOO>"),
    ("MOVSXD~RDX,EAX",                       "LDR~R1,[R11,-0]"),
    ("AND~EAX,0",                            "STMDB~R13!~R7"),
    ("XOR~EDX,EDX",                          "BLS~<FOO>"),
    ("OR~EAX,EAX",                           "SUB~R13,R13,0"),
]

# Evaluate
labels, scores = [], []
oov_pairs = []

print(f"  {'label':>5}  {'cos':>7}  {'x86':<44}  {'arm'}")
print(f"  {'-----':>5}  {'-'*7:>7}  {'-'*44}  {'-'*28}")

for pair_list, label in [(similar, 1), (dissimilar, 0)]:
    tag = "SIM" if label == 1 else "DIS"
    for x86_tok, arm_tok in pair_list:
        v1 = x86_vecs.get(x86_tok)
        v2 = arm_vecs.get(arm_tok)
        if v1 is None or v2 is None:
            oov_pairs.append((tag, x86_tok, arm_tok))
            continue
        s = cosine(v1, v2)
        labels.append(label)
        scores.append(s)
        print(f"  {tag:>5}  {s:+7.4f}  {x86_tok:<44}  {arm_tok}")

if oov_pairs:
    print(f"\n  OOV ({len(oov_pairs)} pairs skipped):")
    for tag, x, a in oov_pairs:
        print(f"    [{tag}]  {x}  vs  {a}")

n_sim = sum(labels)
n_dis = len(labels) - n_sim
print(f"\n  Valid pairs: {len(labels)}  ({n_sim} similar / {n_dis} dissimilar)")

if len(set(labels)) == 2:
    auc = roc_auc_score(labels, scores)
    sim_s = [s for s, l in zip(scores, labels) if l == 1]
    dis_s = [s for s, l in zip(scores, labels) if l == 0]
    print(f"\n  Similar    mean cos = {np.mean(sim_s):+.4f}  min={np.min(sim_s):+.4f}  max={np.max(sim_s):+.4f}")
    print(f"  Dissimilar mean cos = {np.mean(dis_s):+.4f}  min={np.min(dis_s):+.4f}  max={np.max(dis_s):+.4f}")
    print(f"\n  AUC = {auc:.4f}  (paper reports 0.78)")


  label      cos  x86                                           arm
  -----  -------  --------------------------------------------  ----------------------------
    SIM  +0.1936  CALL~RAX                                      BLX~R3
    SIM  +0.1907  CALL~RDX                                      BLX~R4
    SIM  +0.2308  CALL~RBX                                      BLX~R5
    SIM  +0.3529  JMP~RAX                                       BX~R3
    SIM  +0.3404  JGE~<FOO>                                     BGE~<FOO>
    SIM  +0.4240  JL~<FOO>                                      BLT~<FOO>
    SIM  +0.1820  JNS~<FOO>                                     BPL~<FOO>
    SIM  +0.3343  JNC~<FOO>                                     BCC~<FOO>
    SIM  +0.3095  JC~<FOO>                                      BCS~<FOO>
    SIM  +0.2372  JO~<FOO>                                      BVS~<FOO>
    SIM  +0.6970  JZ~<FOO>                                      BEQ~<FOO>
    SIM  +0.6253  JNZ~<FOO>           

In [ ]:
import numpy as np

EMB_BASE  = "/home/tommy/Projects/PCBSDA/experiment/cross-architecture/unimap/embeddings"
SEED_BASE = "/home/tommy/Projects/PCBSDA/experiment/cross-architecture/unimap/seeds"

pairs_cfg = [
    ("x86_64_arm_32",  "x86_64_mapped.vec",  "arm_32_mapped.vec",  "x86_64_arm_32.txt"),
    ("x86_64_mips_32", "x86_64_mapped.vec",  "mips_32_mapped.vec", "x86_64_mips_32.txt"),
    # ("mips_32_arm_32", "mips_32_mapped.vec", "arm_32_mapped.vec",  "mips_32_arm_32.txt"),
]

def load_vec(path):
    words, vecs = [], []
    with open(path, encoding='utf-8') as f:
        n, dim = map(int, f.readline().split())
        for line in f:
            parts = line.rstrip().split(' ')
            words.append(parts[0])
            vecs.append(list(map(float, parts[1:])))
    return {w: np.array(v, dtype=np.float32) for w, v in zip(words, vecs)}

def cosine(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return float('nan')
    return float(np.dot(v1, v2) / (n1 * n2))

for name, src_file, trg_file, seed_file in pairs_cfg:
    src_vecs = load_vec(f"{EMB_BASE}/{name}/{src_file}")
    trg_vecs = load_vec(f"{EMB_BASE}/{name}/{trg_file}")

    rows = []
    with open(f"{SEED_BASE}/{seed_file}") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            rows.append((parts[0], parts[1]))

    print(f"\n{'='*70}")
    print(f"  {name}  ({len(rows)} pairs)")
    print(f"{'='*70}")
    print(f"  {'cos':>7}  {'src':<40}  {'trg'}")
    print(f"  {'-'*7}  {'-'*40}  {'-'*30}")

    sims = []
    for src_tok, trg_tok in rows:
        v1, v2 = src_vecs.get(src_tok), trg_vecs.get(trg_tok)
        if v1 is None or v2 is None:
            tag = "OOV"
            print(f"  {'   OOV':>7}  {src_tok:<40}  {trg_tok}")
        else:
            s = cosine(v1, v2)
            sims.append(s)
            print(f"  {s:+7.4f}  {src_tok:<40}  {trg_tok}")

    sims_arr = np.array([s for s in sims if s == s])
    if len(sims_arr):
        print(f"\n  mean={sims_arr.mean():+.4f}  min={sims_arr.min():+.4f}  "
              f">=0.4: {(sims_arr>=0.4).sum()}/{len(sims_arr)}  "
              f">=0.6: {(sims_arr>=0.6).sum()}/{len(sims_arr)}")



  x86_64_arm_32  (55 pairs)
      cos  src                                       trg
  -------  ----------------------------------------  ------------------------------
  +0.6824  CALL~<FOO>                                BL~<FOO>
  +0.2725  CALL~QWORD~PTR~[RAX+0]                    BLX~R3
  +0.3257  RET                                       BX~R14
  +0.5134  JMP~<FOO>                                 B~<FOO>
  +0.6970  JZ~<FOO>                                  BEQ~<FOO>
  +0.6253  JNZ~<FOO>                                 BNE~<FOO>
  +0.5324  JA~<FOO>                                  BHI~<FOO>
  +0.5514  JBE~<FOO>                                 BLS~<FOO>
  +0.2824  JG~<FOO>                                  BGT~<FOO>
  +0.4441  JLE~<FOO>                                 BLE~<FOO>
  +0.2557  JS~<FOO>                                  BMI~<FOO>
  +0.6140  SUB~RSP,0                                 SUB~R13,R13,0
  +0.5141  ADD~RSP,0                                 ADD~R13,R13,0
  +0.3882  M